### 🎯 Surrogate Model Extraction (Black-Box Evasion)

In this notebook, we will train a **Surrogate Model** to extract the decision boundary of a black-box IDS. 
We do not have access to the dataset, parameters, or type of the defending model. We can only interact with it by querying an API with simulated packet characteristics and receiving a response (`1` for normal, `-1` for anomaly).

In [37]:
import numpy as np
import pandas as pd
import random
import joblib
from sklearn.tree import DecisionTreeClassifier

random.seed(42)
np.random.seed(42)

In [38]:
# Simulating the Defender's Black-Box API
# In a real scenario, this would be a remote system. Here, we load the saved model,
# but we treat it strictly as a black box: we only query it via the function below.

def query_defender_api(packet_size, duration_ms, src_port, protocol="TCP"):
    """
    Simulates a query to the remote black-box IDS.
    Returns:
       1 if the defender classifies the packet as NORMAL.
      -1 if the defender classifies the packet as ANOMALY.
    """
    # Load defender model internally to simulate the API response
    defender_model = joblib.load("anomaly_model.joblib")
    
    # Format the query into a dataframe
    # (Using a simple baseline mapping similar to train_model)
    query_df = pd.DataFrame([{
        "src_port": src_port,
        "dst_port": 8080, 
        "packet_size": packet_size,
        "duration_ms": duration_ms,
        "protocol": protocol
    }])
    
    # We convert to numpy array to match the simple format expected by the model
    # Note: If your preprocessing in task 1 is different, match the format here.
    query_df_processed = pd.get_dummies(query_df, columns=['protocol'], drop_first=True)
    expected_cols = ['src_port', 'dst_port', 'packet_size', 'duration_ms', 'protocol_UDP']
    for col in expected_cols:
        if col not in query_df_processed.columns:
            query_df_processed[col] = 0
            
    query_df_processed = query_df_processed[expected_cols]
    
    prediction = defender_model.predict(query_df_processed)[0]
    return prediction

In [39]:
# Generate random probe inputs to query the black box and build our surrogate dataset.
# We will generate probe samples covering different ranges of packet sizes and durations.
COMMON_PORTS = [80, 443, 22, 8080]

num_probes = 1500
probe_data = []

for _ in range(num_probes):
    src_port = random.choice(COMMON_PORTS)
    packet_size = random.randint(50, 2000)
    duration_ms = random.randint(10, 1000)
    probe_data.append({
        "src_port": src_port,
        "packet_size": packet_size,
        "duration_ms": duration_ms
    })

df_probes = pd.DataFrame(probe_data)

In [40]:
# TODO 1: Query the black box to collect labels for your probe dataset.
# Iterate through df_probes, pass the features to query_defender_api, 
# and record the return labels (1 or -1) in a new column called 'label'.

labels = []
for index, row in df_probes.iterrows():
    label = query_defender_api(
        packet_size=row['packet_size'], 
        duration_ms=row['duration_ms'], 
        src_port=row['src_port']
    )
    labels.append(label)

df_probes['label'] = labels
display(df_probes.head())

,src_port,packet_size,duration_ms,label
0,80,101,769,-1
1,22,551,238,1
2,443,1558,114,1
3,80,1259,442,1
4,80,111,105,1


In [41]:
# TODO 2: Train the Surrogate Model.
# Now that you have a labeled dataset representing the defender's behavior,
# train a local DecisionTreeClassifier to learn the defender's decision boundary.

X_surrogate = df_probes[['src_port', 'packet_size', 'duration_ms']]
y_surrogate = df_probes['label']

# TODO: Initialize and fit a DecisionTreeClassifier on X_surrogate and y_surrogate
surrogate_model = DecisionTreeClassifier(random_state=42)
surrogate_model.fit(X_surrogate, y_surrogate)

# Check how well your surrogate matches the defender's boundary on the training probes
accuracy = surrogate_model.score(X_surrogate, y_surrogate)
print(f"Surrogate model approximation accuracy: {accuracy * 100:.2f}%")

Surrogate model approximation accuracy: 100.00%


In [42]:
# Save the surrogate model, so it can be used by the adversary server
joblib.dump(surrogate_model, "surrogate_model.joblib")
print("Surrogate model saved as surrogate_model.joblib")

Surrogate model saved as surrogate_model.joblib
